### 1.1.8. Ordered Sets

An **ordered set** in AMPL is a one-dimensional set whose members carry an explicit sequence.
The sequence is determined by the order in which members are supplied — either in the data file
or in a model-level literal list — and is preserved throughout the model. Ordering makes a set of
members into a list: each member has a unique position, an immediate predecessor (except the first),
and an immediate successor (except the last). Position-dependent functions — `first`, `last`,
`prev`, `next`, `ord`, `ord0` — are only valid on ordered (or circular) sets.

A **circular** set is an ordered set that wraps: the last member's successor is the first member,
and the first member's predecessor is the last. This is the natural representation for cyclically
repeating entities such as days of the week, months of the year, or shift rotations.

The canonical motivation for ordered sets is the time-indexed optimization model: an inventory
balance constraint links period $t$ to period $t-1$, which requires a well-defined notion of
"the previous period" for every period except the first. The `prev` function provides exactly this
when the period index set is declared `ordered`.

See also: [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) and
[1.1.4. Set Operations](<1_1_4 set_operations.ipynb>).

In [ ]:
# Formal syntax for ordered and circular set declarations
#
# set NAME ordered ;                        -- data-supply order is the sequence
# set NAME ordered = { member-list } ;      -- model-embedded literal list fixes order
# set NAME ordered default { member-list }; -- fallback list; overridable
#
# set NAME circular ;                       -- last wraps to first
# set NAME circular = { member-list } ;     -- literal circular set
#
# set NAME ordered by SUPERSET ;            -- inherit ordering from SUPERSET
# set NAME ordered by reversed SUPERSET ;   -- inherit ordering in reverse
# set NAME circular by SUPERSET ;           -- circular inheritance from SUPERSET
#
# Constraints:
#   - Only one-dimensional sets may be ordered or circular.
#   - SUPERSET in 'ordered by' must itself be an ordered set containing NAME.
#   - 'ordered by' implies 'within SUPERSET' — the subset constraint is automatic.
#
# Position-dependent functions (require an ordered or circular set S):
#   first(S)         -- first member of S
#   last(S)          -- last member of S
#   member(j, S)     -- j-th member of S (1-based, j integer)
#   next(e, S)       -- member immediately after e in S
#   next(e, S, k)    -- member k positions after e in S
#   prev(e, S)       -- member immediately before e in S
#   prev(e, S, k)    -- member k positions before e in S
#   nextw(e, S)      -- next with wraparound (treats S as circular)
#   prevw(e, S)      -- prev with wraparound (treats S as circular)
#   ord(e, S)        -- 1-based ordinal position of e; error if e not in S
#   ord0(e, S)       -- 1-based ordinal position of e; returns 0 if e not in S
#
# When e is a dummy index in scope over S, the set argument may be omitted:
#   {t in S: ord(t) > 1}   is equivalent to   {t in S: ord(t, S) > 1}

#### The `ordered` Keyword

Appending `ordered` to a set declaration instructs AMPL to record the order in which
members are presented. When members are supplied via a data section, AMPL uses the left-to-right
order of the `:=` list as the set's sequence. When members are embedded in the model as a literal
set (`= { ... }`), the enumeration order of the literal list is used.

Without `ordered`, AMPL gives no guarantee about the sequence in which members are enumerated;
the ordering is implementation-defined and may change across builds. Adding `ordered` converts
this unspecified sequence into a contract: member positions are stable, reproducible, and available
to position-dependent functions.

The `ordered` keyword has no effect on which members belong to the set — it affects only the
sequence of existing members. Data validation (`within`, type checks) proceeds identically for
ordered and unordered sets.

In [ ]:
# Minimal ordered set declarations

set WEEKS ordered;                          # members come from data; data-supply order is the sequence
set SHIFTS ordered = {"Day", "Evening", "Night"};  # literal list; order is fixed at model-generation time
set YEARS ordered default {2024, 2025, 2026};      # fallback order; overridable via data or let

# Data file assigns members in desired order:
# data;
# set WEEKS := 27sep 04oct 11oct 18oct ;   # sequence: 27sep < 04oct < 11oct < 18oct

# Immediate use of position functions after ordering is established:
display first(SHIFTS);    # "Day"
display last(SHIFTS);     # "Night"
display member(2, SHIFTS); # "Evening"
display ord("Evening", SHIFTS);  # 2

#### The `circular` Keyword

A `circular` set is ordered and additionally wraps: calling `next` on the last member returns
the first member, and calling `prev` on the first member returns the last. This makes `next` and
`prev` total functions — they never produce a boundary error regardless of which member is passed.

Circular sets model naturally cyclic entities. Days of the week cycle: after Sunday comes Monday.
Months of the year cycle: after December comes January. Scheduling models that rotate resources
through shifts or crew assignments use circular sets to avoid special-casing the wrap-around
period.

In an `ordered` (non-circular) set, calling `next` on the last member or `prev` on the first
member is a runtime error. In a `circular` set the same calls succeed. The wraparound functions
`nextw` and `prevw` provide the same wrap-around behaviour for any ordered set, without
requiring the set to be declared circular.

In [ ]:
# Circular set — last member wraps to first

set DAYS circular = {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"};

display next("Sun", DAYS);   # "Mon" — wraps around (not an error)
display prev("Mon", DAYS);   # "Sun" — wraps around (not an error)
display ord("Fri", DAYS);    # 5

# For an ordered (non-circular) set, the same calls at the boundary are runtime errors:
set DAYS2 ordered = {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"};
# next("Sun", DAYS2)  -- ERROR: 'Sun' is the last member of DAYS2
# prev("Mon", DAYS2)  -- ERROR: 'Mon' is the first member of DAYS2

# nextw/prevw: wraparound versions of next/prev that work on any ordered set
display nextw("Sun", DAYS2); # "Mon" — same as next on a circular set
display prevw("Mon", DAYS2); # "Sun"

#### Position-Dependent Functions

The following table documents every function that requires its set argument to be ordered or
circular. All functions are listed in AMPL Reference Manual Table A-5. In each entry, `S` is an
ordered or circular set of $n$ members, $e$ is the $j$-th member ($1 \le j \le n$), and $k$ is
a positive integer.

| Function | Returns | Boundary behaviour |
|---|---|---|
| `first(S)` | First member of `S` | Always defined |
| `last(S)` | Last member of `S` | Always defined |
| `member(j, S)` | The $j$-th member of `S` (1-based) | Error if $j < 1$ or $j > n$ |
| `next(e, S)` | Member at position $j+1$ | Error at $j=n$ if `S` is ordered; wraps if circular |
| `next(e, S, k)` | Member at position $j+k$ | Error if $j+k > n$ (ordered); wraps if circular |
| `nextw(e, S)` | Member at position $j+1$, wrapping | Always defined; treats `S` as circular |
| `prev(e, S)` | Member at position $j-1$ | Error at $j=1$ if `S` is ordered; wraps if circular |
| `prev(e, S, k)` | Member at position $j-k$ | Error if $j-k < 1$ (ordered); wraps if circular |
| `prevw(e, S)` | Member at position $j-1$, wrapping | Always defined; treats `S` as circular |
| `ord(e, S)` | 1-based position of `e` in `S` | Error if `e` not in `S` |
| `ord0(e, S)` | 1-based position of `e` in `S` | Returns `0` if `e` not in `S` |

**Implicit set argument.** When `e` is a dummy index whose associated indexing set is `S`, the set
argument may be omitted: `ord(t)` inside `{t in WEEKS: ...}` is equivalent to `ord(t, WEEKS)`.
This abbreviation applies to `next`, `prev`, `nextw`, `prevw`, and `ord`.

**Negative and zero step values.** The step argument $k$ in `next(e, S, k)` may be zero or
negative: `next(e, S, 0)` returns `e` itself; `next(e, S, -3)` is identical to `prev(e, S, 3)`.

In [ ]:
# Demonstrating all position-dependent functions on a single ordered set

set MONTHS ordered = {"Jan", "Feb", "Mar", "Apr", "May", "Jun",
                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"};

display first(MONTHS);           # "Jan"
display last(MONTHS);            # "Dec"
display member(6, MONTHS);       # "Jun"

display next("Mar", MONTHS);     # "Apr"  (1 step forward)
display next("Mar", MONTHS, 3);  # "Jun"  (3 steps forward)
display prev("Oct", MONTHS);     # "Sep"  (1 step back)
display prev("Oct", MONTHS, 4);  # "Jun"  (4 steps back)

display ord("Jan", MONTHS);      # 1  (first position)
display ord("Dec", MONTHS);      # 12 (last position)
display ord0("Jan", MONTHS);     # 1  (same as ord when member is present)
display ord0("Xyz", MONTHS);     # 0  (not present — no error)

# Wraparound versions (work on ordered sets without circular declaration)
display nextw("Dec", MONTHS);    # "Jan" — wraps
display prevw("Jan", MONTHS);    # "Dec" — wraps

# Implicit set argument: inside an indexing expression over MONTHS,
# ord(m) and prev(m) implicitly refer to MONTHS.
# Example usage in a constraint:
param budget {MONTHS} >= 0 default 1000;
param carry_rate >= 0 default 0.02;
# Cumulative carry: each month's budget = previous month's budget * (1 + carry_rate)
# subject to CarryForward {m in MONTHS: ord(m) > 1}:
#     budget[m] = budget[prev(m)] * (1 + carry_rate);

#### The `ordered by` Phrase: Inheriting Order from a Superset

When a subset is defined by a filtering expression over an ordered set, the filtered result is
unordered by default — the ordering of the superset is not automatically inherited. To make the
subset ordered in the same sequence as its containing set, use `ordered by SUPERSET` in the
subset's declaration.

The phrase `ordered by SUPERSET` has two simultaneous effects:

1. **Ordering**: members of the declared set appear in the same relative sequence as they do in
   `SUPERSET`. The declared set becomes a fully ordered set; all position-dependent functions
   apply to it.
2. **Membership constraint**: the declared set is implicitly constrained to be a subset of
   `SUPERSET`. This is equivalent to adding `within SUPERSET` to the declaration, so `within`
   need not be stated separately.

The phrase `ordered by reversed SUPERSET` applies the same rules but reverses the inherited
sequence. `circular by SUPERSET` inherits the ordering as a circular set.

A common pattern is a rolling-horizon model: `ALL_WEEKS ordered` holds all available periods,
and `WEEKS ordered by ALL_WEEKS` is a subset covering the active horizon. The subset is
automatically ordered, and `prev`, `first`, and `ord` work on it without further declaration.

In [ ]:
# ordered by: inheriting ordering from a containing ordered set

set ALL_WEEKS ordered;          # master calendar — all available weeks
param T > 0 integer;            # horizon length: use only the first T weeks

# WEEKS is ordered with the same sequence as ALL_WEEKS; within ALL_WEEKS is implied
set WEEKS = {t in ALL_WEEKS: ord(t) <= T} ordered by ALL_WEEKS;

# Both first() and prev() are valid on WEEKS:
display first(WEEKS);           # first week in the active horizon
display last(WEEKS);            # last week in the active horizon

# Rolling-horizon variant: LATE is the final K periods
param K > 0 integer;
set LATE = {t in ALL_WEEKS: ord(t) > card(ALL_WEEKS) - K} ordered by ALL_WEEKS;

# Reversed ordering — useful for backward induction
set WEEKS_REV = {t in ALL_WEEKS: ord(t) <= T} ordered by reversed ALL_WEEKS;
# first(WEEKS_REV) is the T-th week of ALL_WEEKS (last chronologically)

# Exercise 5-6 pattern from Fourer, Gay, Kernighan:
# Merge Balance0 + Balance by excluding the first member from the sub-set
set WEEKS0 ordered;  # data-supplied full calendar
set WEEKS_TAIL = {i in WEEKS0: ord(i) > 1} ordered by WEEKS0;
# prev(t, WEEKS_TAIL) is valid for all t except first(WEEKS_TAIL)

#### `diff` Preserves Ordering; `union`, `inter`, `symdiff` Do Not

The AMPL Reference Manual (A.6.2) specifies the ordering behaviour of set operators applied
to ordered sets:

- **`diff`**: the result is ordered if and only if the **left operand** is ordered. The members
  of the result appear in the same sequence as they do in the left operand. Position-dependent
  functions are valid on the result.
- **`union`, `inter`, `symdiff`**: the ordering of the result is **not defined**; AMPL treats
  the result as unordered even if both operands are ordered. Calling `first`, `prev`, or `ord`
  on such a result is a model error.

The practical consequence is that the standard idiom for excluding the boundary member from an
ordered set in a constraint's indexing expression is to use `diff`:

```ampl
subject to Balance {p in PROD, t in WEEKS diff {first(WEEKS)}}:
    Make[p,t] + Inv[p,prev(t)] = Sell[p,t] + Inv[p,t];
```

Because `diff` preserves the ordering of `WEEKS`, `prev(t)` is valid inside this constraint.
The alternative guard `{t in WEEKS: ord(t) > 1}` is equivalent and equally common.

In [ ]:
# diff preserves ordering; union/inter/symdiff do not

set ALL_WEEKS ordered;          # ordered: {"W1", "W2", "W3", "W4", "W5"} from data
param T integer > 0;

# diff: result is ordered (inherits ordering of ALL_WEEKS)
set WEEKS_NO_LAST = ALL_WEEKS diff {last(ALL_WEEKS)};   # ordered — prev() is valid
set WEEKS_NO_FIRST = ALL_WEEKS diff {first(ALL_WEEKS)}; # ordered — prev() is valid

# These two constraint forms are equivalent — both are valid:
#   Form A: diff-based indexing
#   subject to Bal_A {p in PROD, t in ALL_WEEKS diff {first(ALL_WEEKS)}}:
#       ... prev(t, ALL_WEEKS) ...
#
#   Form B: ord-guard indexing
#   subject to Bal_B {p in PROD, t in ALL_WEEKS: ord(t) > 1}:
#       ... prev(t) ...

# union of two ordered sets: result is UNORDERED
set EARLY ordered by ALL_WEEKS = {t in ALL_WEEKS: ord(t) <= T};
set LATE  ordered by ALL_WEEKS = {t in ALL_WEEKS: ord(t) >  T};

set ALL_COMBINED = EARLY union LATE;    # unordered — first(ALL_COMBINED) is a MODEL ERROR

# Correct alternative: use 'ordered by ALL_WEEKS' on a set covering the full range
set ALL_ORDERED ordered by ALL_WEEKS = {t in ALL_WEEKS: 1 = 1};  # all members, ordered

#### Example 1 — Minimal Ordered Set: Shift Scheduling

This minimal example demonstrates the three essential ordered-set declarations and the
boundary-safe pattern for referencing the predecessor period. It uses a small shift model
where carry-over from the previous shift is tracked.

In [ ]:
# Example 1: Ordered set of named shifts with predecessor reference

set SHIFTS ordered = {"Morning", "Afternoon", "Night"};  # fixed order in model

param demand {SHIFTS} >= 0;  # minimum staff needed in each shift
param carryover_rate >= 0;   # fraction of staff that can roll over from previous shift

var Staff {SHIFTS} >= 0;     # number of staff assigned to each shift

minimize TotalStaff:
    sum {s in SHIFTS} Staff[s];

# First shift: no predecessor; demand must be met directly
subject to DemandFirst:
    Staff[first(SHIFTS)] >= demand[first(SHIFTS)];

# Subsequent shifts: predecessor's staff partially carries over
subject to DemandCarry {s in SHIFTS diff {first(SHIFTS)}}:
    Staff[s] + carryover_rate * Staff[prev(s)] >= demand[s];

# ord() can be used in parameters and conditional expressions
param shift_number {s in SHIFTS} = ord(s);  # Morning=1, Afternoon=2, Night=3
display shift_number;  # Expected: Morning 1, Afternoon 2, Night 3

#### Example 2 — Circular Set: Weekly Crew Rotation

This example shows a circular set used to model a repeating weekly crew rotation schedule.
Each crew works one day on, then hands off to the next crew; after the last crew comes the
first again. The `circular` declaration eliminates the boundary special-case.

In [ ]:
# Example 2: Circular set for cyclic crew rotation

set DAYS circular = {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"};
set CREWS;  # crew identifiers, e.g., {"A", "B", "C"}

param crew_needed {DAYS} >= 0;              # minimum crew count required per day
param rest_days {CREWS} >= 0 integer;       # number of consecutive rest days required after work

var Assign {CREWS, DAYS} binary;            # 1 if crew c is assigned on day d

# Each day must have enough crews assigned
subject to Coverage {d in DAYS}:
    sum {c in CREWS} Assign[c, d] >= crew_needed[d];

# No crew can work if any of the previous rest_days[c] days was also worked
# next/prev wrap around automatically on a circular set:
subject to RestRequired {c in CREWS, d in DAYS, k in 1..rest_days[c]}:
    Assign[c, d] + Assign[c, prev(d, DAYS, k)] <= 1;

# Verifying wrap-around:
display next("Sun", DAYS);   # "Mon" — circular wrap, no error
display prev("Mon", DAYS);   # "Sun" — circular wrap, no error
display ord("Wed", DAYS);    # 3

#### Example 3 — Multi-Period Inventory Balance (The Canonical Ordered-Set Pattern)

This is the model from Fourer, Gay, and Kernighan Chapter 5 (steelT2.mod), the authoritative
reference example for ordered sets in AMPL. It replaces a numeric time index `1..T` with an
ordered set of meaningful week labels, and replaces the expression `t-1` with `prev(t)`. The
first-period boundary is handled by a separate `Balance0` constraint.

The model illustrates:
- `set WEEKS ordered` with string members supplied from data
- `first(WEEKS)` to identify the boundary period
- `{t in WEEKS: ord(t) > 1}` to exclude the first period from the general constraint
- `prev(t)` (implicit set: `WEEKS`) inside the indexed constraint

In [ ]:
# Example 3: steelT2.mod — multi-period steel production (Fourer, Gay, Kernighan, Figure 5-3)

set PROD;           # products (e.g., bands, coils)
set WEEKS ordered;  # weeks — data-supply order is the sequence

param rate    {PROD}        > 0;   # tons per hour produced
param inv0    {PROD}       >= 0;   # initial inventory (tons) at start of first week
param avail   {WEEKS}      >= 0;   # production hours available each week
param market  {PROD,WEEKS} >= 0;   # upper bound on tons sold each week
param prodcost {PROD}      >= 0;   # production cost per ton
param invcost  {PROD}      >= 0;   # inventory carrying cost per ton per week
param revenue  {PROD,WEEKS} >= 0;  # revenue per ton sold

var Make {PROD, WEEKS}             >= 0;              # tons produced
var Inv  {PROD, WEEKS}             >= 0;              # tons held in inventory
var Sell {p in PROD, t in WEEKS}   >= 0, <= market[p,t]; # tons sold

maximize TotalProfit:
    sum {p in PROD, t in WEEKS} (
        revenue[p,t]  * Sell[p,t]
      - prodcost[p]   * Make[p,t]
      - invcost[p]    * Inv[p,t]
    );

# Capacity constraint: total production hours per week cannot exceed available hours
subject to Time {t in WEEKS}:
    sum {p in PROD} (1 / rate[p]) * Make[p,t] <= avail[t];

# First-period inventory balance: initial inventory replaces prev-week inventory
subject to Balance0 {p in PROD}:
    Make[p, first(WEEKS)] + inv0[p]
    = Sell[p, first(WEEKS)] + Inv[p, first(WEEKS)];

# General inventory balance: prev(t) is valid because ord(t) > 1 excludes the first period
subject to Balance {p in PROD, t in WEEKS: ord(t) > 1}:
    Make[p,t] + Inv[p, prev(t)] = Sell[p,t] + Inv[p,t];

# Data section (representative):
# data;
# set PROD  := bands coils ;
# set WEEKS := 27sep 04oct 11oct 18oct ;  # order here is the set's order
# param rate    :=  bands 200  coils 140 ;
# param inv0    :=  bands 10   coils 0   ;
# param avail   :=  27sep 40  04oct 40  11oct 32  18oct 40 ;
# ...

#### Example 3b — Merged Balance Constraint Using `if-then-else`

The two-constraint pattern (`Balance0` + `Balance`) can be merged into a single declaration
using AMPL's `if-then-else` expression. The conditional selects between the initial inventory
parameter `inv0[p]` for the first period and the variable `Inv[p, prev(t)]` for all others.

Both forms are correct; the merged form is more concise. The two-constraint form is more
explicit about boundary handling and may be easier to debug.

In [ ]:
# Example 3b: Merged balance constraint (Fourer, Gay, Kernighan, Chapter 8 discussion)

# Same declarations as Example 3 (PROD, WEEKS, Make, Inv, Sell, inv0 assumed declared above)

# Single constraint covering all weeks; the conditional handles the boundary
subject to Balance_merged {p in PROD, t in WEEKS}:
    Make[p,t]
    + (if t = first(WEEKS) then inv0[p] else Inv[p, prev(t)])
    = Sell[p,t] + Inv[p,t];

# Note: prev(t) inside the else branch is safe because AMPL evaluates the expression
# lazily — it is never called when t = first(WEEKS).  However, the two-constraint
# form (Balance0 + Balance with ord(t) > 1 guard) is often considered clearer
# for documentation purposes.

#### Example 4 — Rolling Horizon with `ordered by`

This example demonstrates the `ordered by` pattern for variable-length planning horizons.
A master set `ALL_WEEKS` contains all possible periods; the active subset `WEEKS` contains
only the first `T` periods and inherits the ordering of `ALL_WEEKS`.

In [ ]:
# Example 4: Rolling horizon with 'ordered by' (Fourer, Gay, Kernighan, Section 5.6)

set ALL_WEEKS ordered;          # full calendar — data-supplied; e.g., 52 weeks
param T > 0 integer;            # active horizon length (from data)

# WEEKS is ordered (same sequence as ALL_WEEKS) and implicitly within ALL_WEEKS
set WEEKS = {t in ALL_WEEKS: ord(t) <= T} ordered by ALL_WEEKS;

# Now WEEKS is a fully ordered set; all position functions apply:
# first(WEEKS) = first(ALL_WEEKS)  [they share the same starting member]
# last(WEEKS)  = member(T, ALL_WEEKS)

set PROD2;
param inv0_r {PROD2} >= 0;
param avail_r {ALL_WEEKS} >= 0;  # indexed over full calendar; WEEKS is a subset
param rate_r  {PROD2} > 0;

var Make_r {PROD2, WEEKS} >= 0;
var Inv_r  {PROD2, WEEKS} >= 0;
var Sell_r {PROD2, WEEKS} >= 0;

subject to Time_r {t in WEEKS}:
    sum {p in PROD2} (1 / rate_r[p]) * Make_r[p,t] <= avail_r[t];

subject to Bal0_r {p in PROD2}:
    Make_r[p, first(WEEKS)] + inv0_r[p]
    = Sell_r[p, first(WEEKS)] + Inv_r[p, first(WEEKS)];

subject to Bal_r {p in PROD2, t in WEEKS: ord(t) > 1}:
    Make_r[p,t] + Inv_r[p, prev(t)] = Sell_r[p,t] + Inv_r[p,t];

# Changing the horizon: supply a different value of T in the data file.
# The model is unchanged; WEEKS shrinks or grows automatically.
# data;
# set ALL_WEEKS := W1 W2 W3 W4 W5 W6 W7 W8;
# param T := 4;

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
In a time-indexed model, declare a master ordered set (e.g., <code>ALL_WEEKS ordered</code>)
and derive active subsets from it using <code>ordered by ALL_WEEKS</code>. This keeps ordering
consistent across all subsets, allows the horizon length to be controlled by a single parameter,
and avoids re-declaring ordering on every derived set individually.
</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use <code>circular</code> only when the physical entity is genuinely cyclic (days of the week,
months of the year, shift rotations). For time-series models where the sequence has a definite
start and end, use <code>ordered</code> and guard boundary calls with <code>ord(t) > 1</code>
or <code>t = first(S)</code>. Using <code>circular</code> on a non-cyclic set silently produces
incorrect models: calling <code>prev</code> on the first period wraps to the last instead of
producing an error.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Calling <code>prev(t, S)</code> or <code>next(t, S)</code> when <code>t</code> is the first or
last member of an <code>ordered</code> set is a <strong>runtime error</strong>, not a model error.
AMPL reports it at constraint-generation time, not at declaration time. The error message is:
<em>"can't compute prev('27sep', WEEKS) — '27sep' is the first member"</em>. Guard boundary
calls with <code>ord(t) &gt; 1</code> (for <code>prev</code>) or <code>ord(t) &lt; card(S)</code>
(for <code>next</code>), or use a separate first/last constraint.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The result of <code>union</code>, <code>inter</code>, and <code>symdiff</code> on two ordered sets
is <strong>unordered</strong> even if both operands are ordered (AMPL Reference Manual A.6.2).
Only <code>diff</code> preserves the ordering of its left operand. Calling <code>first</code>,
<code>prev</code>, or <code>ord</code> on a <code>union</code> or <code>inter</code> result is a
model error. To obtain an ordered result from a combination of sets, use
<code>ordered by SUPERSET</code> on a set whose members are derived from the superset.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Sets of dimension two or higher <strong>cannot be declared <code>ordered</code> or
<code>circular</code></strong>. This restriction is absolute: a set of pairs, triples, or any
tuple of length two or more is always unordered, regardless of how it was constructed.
Attempting <code>set LINKS ordered within ORIG cross DEST</code> is a syntax error.
If you need position-dependent access to component sets, declare each component set
ordered separately.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A set defined with <code>=</code> from an arithmetic progression expression
(<code>set YEARS = 2020..2030;</code>) is <strong>not automatically ordered</strong> unless the
declaration includes <code>ordered</code>. The range expression is manifestly ordered internally,
and AMPL may use that ordering for <code>diff</code> operations, but <code>first(YEARS)</code>
and <code>prev(t, YEARS)</code> remain model errors without the explicit <code>ordered</code>
keyword. Write <code>set YEARS ordered = 2020..2030;</code> to make the set ordered.
</div>

#### amplpy: Working with Ordered Sets

In amplpy, ordered set members are supplied and retrieved as Python lists. Because a Python
list preserves insertion order, assigning `ampl.set["WEEKS"] = weeks_list` delivers members
to AMPL in exactly the order they appear in `weeks_list`, which becomes the set's ordering.

The amplpy `Set` object does not expose position-dependent functions directly — `first`,
`prev`, `ord`, and so on are AMPL-level functions and must be invoked via `ampl.eval()` or
`ampl.get_value()`. To retrieve members in sequence, use `ampl.set[name].members()`, which
returns a Python list in the set's declared order.

| Task | amplpy pattern |
|---|---|
| Assign ordered members | `ampl.set["WEEKS"] = ["27sep", "04oct", "11oct", "18oct"]` |
| Retrieve members in order | `ampl.set["WEEKS"].members()` — returns a list in declared order |
| Get first member (AMPL) | `ampl.get_value("first(WEEKS)")` |
| Get ordinal position | `ampl.get_value("ord('04oct', WEEKS)")` |
| Evaluate prev/next | `ampl.eval("display prev('04oct', WEEKS);")` |
| Cardinality | `ampl.set["WEEKS"].size()` |
| Read members as DataFrame | `ampl.set["WEEKS"].get_values().to_pandas()` |

In [ ]:
# amplpy: supplying and retrieving ordered set members

from amplpy import AMPL

ampl = AMPL()

ampl.eval("""
    set PROD;          # products
    set WEEKS ordered; # weeks — order is determined by data-supply order
""")

# Supply members in the desired order via a Python list.
# AMPL records this left-to-right order as the set's sequence.
ampl.set["PROD"]  = ["bands", "coils"]
ampl.set["WEEKS"] = ["27sep", "04oct", "11oct", "18oct"]  # order: 27sep < 04oct < ...

# Retrieve members back in the declared order (not necessarily insertion order
# for unordered sets, but for ordered sets the list matches the declared sequence):
weeks_in_order = ampl.set["WEEKS"].members()
print(weeks_in_order)  # ['27sep', '04oct', '11oct', '18oct']

# Invoke AMPL-level position functions via get_value:
first_week = ampl.get_value("first(WEEKS)")
print(first_week)   # 27sep

last_week = ampl.get_value("last(WEEKS)")
print(last_week)    # 18oct

# ord() returns a float from get_value; convert to int as needed
pos = int(ampl.get_value("ord('04oct', WEEKS)"))
print(pos)          # 2

# prev/next via eval (result printed by AMPL)
ampl.eval("display next('04oct', WEEKS);")
ampl.eval("display prev('11oct', WEEKS);")

# Iterate over members in order from Python
for i, w in enumerate(ampl.set["WEEKS"].members(), start=1):
    print(f"  Position {i}: {w}")

#### Edge Cases

**Edge case 1: `prev` on the first member of an `ordered` set is a runtime error, not a syntax error.**

AMPL does not detect this mistake at model-generation time. The model compiles without error.
The error occurs when AMPL generates the constraint instances and evaluates `prev(t)` with `t`
bound to the first member. The fix is either an `ord(t) > 1` guard or a separate first-period
constraint.

In [ ]:
# Edge case 1: runtime error from calling prev at the boundary

set WEEKS_EC1 ordered;
var x {WEEKS_EC1} >= 0;

# WRONG — model compiles, but generating constraints raises an error:
# subject to Link_wrong {t in WEEKS_EC1}:
#     x[t] >= x[prev(t)];  # ERROR at t = first(WEEKS_EC1): 'prev' undefined

# FIX A: guard with ord(t) > 1
subject to Link_A {t in WEEKS_EC1: ord(t) > 1}:
    x[t] >= x[prev(t)];   # valid: first member excluded

# FIX B: separate constraint for the first period
param x0 >= 0;  # initial value
subject to Link_B0:
    x[first(WEEKS_EC1)] >= x0;
subject to Link_B {t in WEEKS_EC1: ord(t) > 1}:
    x[t] >= x[prev(t)];

# FIX C: use diff to exclude the first member from the index set
subject to Link_C {t in WEEKS_EC1 diff {first(WEEKS_EC1)}}:
    x[t] >= x[prev(t)];   # diff preserves ordering, so prev(t) is valid

**Edge case 2: An arithmetic progression expression is manifestly ordered, but the named set requires an explicit `ordered` declaration.**

The expression `T_start .. T_end` is internally ordered in AMPL, but a named set defined by
this expression without an `ordered` keyword is officially unordered. Position-dependent functions
cannot be applied to the named set. The `ordered` keyword must appear in the declaration.

In [ ]:
# Edge case 2: arithmetic progression — manifest ordering vs. declared ordering

param T_start integer default 2020;
param T_end   integer > T_start default 2025;

# Without 'ordered': YEARS is defined by an ordered expression but is itself unordered
set YEARS = T_start .. T_end;    # {2020, 2021, 2022, 2023, 2024, 2025}
# first(YEARS) is a MODEL ERROR — 'YEARS' is not declared ordered

# With 'ordered': YEARS2 is explicitly ordered; all position functions are valid
set YEARS2 ordered = T_start .. T_end;
display first(YEARS2);           # 2020
display last(YEARS2);            # 2025
display next(2022, YEARS2);      # 2023
display prev(2024, YEARS2);      # 2023
display ord(2022, YEARS2);       # 3

# Note: 'diff' on an unordered set whose expression is an arithmetic progression:
# AMPL retains the manifest ordering in the result, so first()/prev() MAY work on
# (YEARS diff {2025}) even without 'ordered', but this is implementation-defined
# behaviour and should NOT be relied upon. Always declare 'ordered' explicitly.

**Edge case 3: `union` of two `ordered` sets produces an unordered result — calling `first` on it is a model error.**

This is one of the most common mistakes in rolling-horizon models. A modeler expects
`EARLY union LATE` to cover all periods in order, but the result is unordered. The correct
approach is to define the covering set as `ordered by ALL_WEEKS`.

In [ ]:
# Edge case 3: union of ordered sets yields unordered result

set ALL ordered;             # master ordered set, e.g., 8 weeks
param Split integer > 0;     # divides ALL into two halves

set FIRST_HALF ordered by ALL = {t in ALL: ord(t) <= Split};
set SECOND_HALF ordered by ALL = {t in ALL: ord(t) > Split};

# WRONG: union result is unordered
set COMBINED = FIRST_HALF union SECOND_HALF;  # unordered!
# first(COMBINED)  -- MODEL ERROR: 'COMBINED' is not ordered
# prev(t, COMBINED) -- MODEL ERROR: 'COMBINED' is not ordered

# CORRECT: use 'ordered by ALL' on a set that covers all members
set COMBINED_ORDERED = {t in ALL: 1 = 1} ordered by ALL;
display first(COMBINED_ORDERED);    # valid
display last(COMBINED_ORDERED);     # valid

# Alternative correct approach: diff preserves ordering
set ALL_EXCEPT_LAST ordered by ALL = ALL diff {last(ALL)};
display first(ALL_EXCEPT_LAST);     # valid — diff preserves ordering of left operand

**Edge case 4: `ord0` as a safe membership check that also returns position.**

`ord(e, S)` raises an error if `e` is not a member of `S`. `ord0(e, S)` returns 0 instead.
This makes `ord0` useful in two roles: (a) testing membership without a separate `in` check,
and (b) generating constraints only for elements that appear in the set.

In [ ]:
# Edge case 4: ord vs. ord0 for safe membership and position lookup

set PRIORITY_WEEKS ordered;  # a subset of all weeks that are "priority" periods
set ALL_WEEKS2 ordered;      # all periods

param base_cost {ALL_WEEKS2} >= 0;
param priority_surcharge >= 0;

# ord0 returns 0 for non-members — safe to use in a conditional expression
param effective_cost {t in ALL_WEEKS2} =
    base_cost[t]
    + (if ord0(t, PRIORITY_WEEKS) > 0 then priority_surcharge else 0);

# ord would raise an error for any t not in PRIORITY_WEEKS:
# param wrong_cost {t in ALL_WEEKS2} =
#     if ord(t, PRIORITY_WEEKS) > 0 then ...   # ERROR if t not in PRIORITY_WEEKS

# ord0 as a rank indicator within the priority set
param priority_rank {t in PRIORITY_WEEKS} = ord(t, PRIORITY_WEEKS);  # 1-based rank
# or equivalently: ord0(t, PRIORITY_WEEKS)  -- same result when t is in PRIORITY_WEEKS

display priority_rank;  # prints the 1-based position of each priority week

**Edge case 5: Applying `ordered by` without matching the superset constraint.**

If the expression defining the subset produces a member that is not in the superset named in
`ordered by`, AMPL raises an error at model-generation time. This is equivalent to a `within`
violation, because `ordered by SUPERSET` implies `within SUPERSET`.

In [ ]:
# Edge case 5: ordered by enforces subset membership

set MASTER ordered;     # e.g., {"W1", "W2", "W3", "W4"}

# CORRECT: the expression draws only from MASTER; within MASTER is implied
param T2 integer > 0;
set SUB = {t in MASTER: ord(t) <= T2} ordered by MASTER;

# ERROR scenario: if SUB were assigned a member not in MASTER, AMPL would report
# a membership violation at generation time.
# Example triggering the error:
# let SUB := SUB union {"W99"};   # ERROR: 'W99' is not in MASTER
#
# The 'within MASTER' constraint is enforced by 'ordered by MASTER';
# no separate 'within' phrase is needed or permitted in the same declaration.

# Valid: both 'ordered by MASTER' and filtering within the expression
set FIRST_THREE = {t in MASTER: ord(t) <= 3} ordered by MASTER;
display first(FIRST_THREE);  # first(MASTER)
display card(FIRST_THREE);   # min(3, card(MASTER))

#### Summary of Ordered and Circular Set Attributes

| Declaration phrase | Effect | Within constraint | Position functions |
|---|---|---|---|
| `set S ordered;` | Data-supply order is the sequence | None | All valid; boundary errors at first/last |
| `set S circular;` | Data-supply order; last wraps to first | None | All valid; no boundary errors |
| `set S ordered by T;` | Inherits order of `T`; `T` must be ordered | `S within T` implied | All valid; boundary errors at first/last |
| `set S ordered by reversed T;` | Inherits reverse order of `T` | `S within T` implied | All valid; first/last are reversed |
| `set S circular by T;` | Inherits order of `T`; wraps | `S within T` implied | All valid; no boundary errors |

See also:
- [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — the default set type; functions not valid on unordered sets
- [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — ordering preservation rules for `diff`, `union`, `inter`, `symdiff`
- [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — `=`, `:=`, `default` in set declarations